In [ ]:
# Célula 1 — instalar
!pip install jupysql pandas openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.1/95.1 kB 928.6 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.8/192.8 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.0/264.0 kB 5.2 MB/s eta 0:00:00


In [ ]:
# importing all dependencies


import pandas as pd
import sqlite3
from google.colab import drive
import os

In [ ]:
# loading google colab and creating save point on google drive
drive.mount('/content/drive')
OUTPUT_DIR = "/content/drive/MyDrive/MozDev_AI_Automation_Scripts/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

Mounted at /content/drive


In [ ]:
# creating a path to drive
cd "drive/MyDrive/MozDev_AI_Automation_Scripts/"

/content/drive/MyDrive/MozDev_AI_Automation_Scripts


In [ ]:
ls

 fact_mozbus.xlsx                   MozChapa100_Vendas_ERROS.xlsx
 mozchapa100.db                    'MozDev Script Automation Part 1'
 MozChapa100_GABARITO.xlsx         'MozDev Script Automation Road Map 06 2026'
 MozChapa100_Rotas_ERROS.xlsx      'Part 1 Template_MozDev202606'
 MozChapa100_Segmentos_ERROS.xlsx   Template_MozDev202606.ipynb


In [ ]:
"""
MozChapa100 — Gerador de Ficheiros Excel COM ERROS

Erros injectados (apenas 3 tipos):
  1. NEGATIVO     — valores numéricos negativos
  2. ESPACOS      — espaços extras no método de pagamento  (" M-Pesa ")
  3. DATA_ERRADA  — data com hora incluída (formato errado: datetime em vez de date)

Ficheiros gerados:
  - MozChapa100_Segmentos_ERROS.xlsx
  - MozChapa100_Vendas_ERROS.xlsx
  - MozChapa100_Rotas_ERROS.xlsx
"""

import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from datetime import date, datetime, timedelta
import random

random.seed(42)

OUTPUT_DIR = ""

COR_HEADER = "1F4E79"
COR_SUB    = "2E75B6"
COR_PAR    = "D6E4F0"
COR_TOTAL  = "1F4E79"

# ============================================================
# ESTILOS
# ============================================================

def borda():
    s = Side(style="thin", color="AAAAAA")
    return Border(left=s, right=s, top=s, bottom=s)

def hdr(cell, bg=COR_SUB):
    cell.font = Font(bold=True, color="FFFFFF", name="Arial", size=10)
    cell.fill = PatternFill("solid", start_color=bg)
    cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    cell.border = borda()

def dado(cell, par=False):
    cell.font = Font(name="Arial", size=10)
    cell.fill = PatternFill("solid", start_color=COR_PAR if par else "FFFFFF")
    cell.alignment = Alignment(horizontal="center", vertical="center")
    cell.border = borda()

def adicionar_titulo(ws, texto, n_colunas):
    ws.merge_cells(f"A1:{get_column_letter(n_colunas)}1")
    c = ws["A1"]
    c.value = texto
    c.font = Font(bold=True, color="FFFFFF", name="Arial", size=13)
    c.fill = PatternFill("solid", start_color=COR_HEADER)
    c.alignment = Alignment(horizontal="center", vertical="center")
    ws.row_dimensions[1].height = 30

def adicionar_total(ws, linha, col_ate, col_soma, ultima):
    ws.merge_cells(f"A{linha}:{get_column_letter(col_ate)}{linha}")
    ct = ws.cell(row=linha, column=1, value="TOTAL GERAL")
    ct.font = Font(bold=True, color="FFFFFF", name="Arial", size=11)
    ct.fill = PatternFill("solid", start_color=COR_TOTAL)
    ct.alignment = Alignment(horizontal="right", vertical="center")
    ct.border = borda()
    col_l = get_column_letter(col_soma)
    cs = ws.cell(row=linha, column=col_soma, value=f"=SUM({col_l}3:{col_l}{ultima})")
    cs.font = Font(bold=True, color="FFFFFF", name="Arial", size=11)
    cs.fill = PatternFill("solid", start_color=COR_TOTAL)
    cs.alignment = Alignment(horizontal="center", vertical="center")
    cs.border = borda()
    if col_soma in [6, 7, 11]:
        cs.number_format = '#,##0.00" MZN"'

def fill_total_vazio(ws, linha, colunas):
    for col in colunas:
        ec = ws.cell(row=linha, column=col, value="")
        ec.fill = PatternFill("solid", start_color=COR_TOTAL)
        ec.border = borda()

# ============================================================
# INJECTAR ERROS — apenas 3 tipos
# ============================================================

def injectar_erro(valor, tipo_erro):
    if tipo_erro == "negativo" and isinstance(valor, (int, float)):
        return -abs(valor)
    elif tipo_erro == "espacos" and isinstance(valor, str):
        return f" {valor} "
    elif tipo_erro == "data_errada" and isinstance(valor, date):
        # Converte date para datetime com hora aleatória — erro de formato
        hora = random.randint(0, 23)
        minuto = random.randint(0, 59)
        return datetime(valor.year, valor.month, valor.day, hora, minuto, 0)
    return valor

# ============================================================
# DATAS
# ============================================================

datas = []
d = date(2026, 3, 1)
while d <= date(2026, 5, 31):
    datas.append(d)
    d += timedelta(days=1)

erros_log = {
    "Segmentos": [],
    "Vendas": [],
    "Rotas": []
}

# ============================================================
# FILE 1 — SEGMENTOS
# ============================================================

wb1 = openpyxl.Workbook()
ws1 = wb1.active
ws1.title = "Segmentos"

adicionar_titulo(ws1, "MOZCHAPA100 — SEGMENTOS DE PASSAGEIROS (Mar–Mai 2026)", 9)
ws1.row_dimensions[2].height = 36

cabs1 = [
    "ID_Segmento", "Data", "Tipo_Passageiro", "Genero",
    "Faixa_Etaria", "Total_Passageiros", "Hora_Pico",
    "Pagamento_Preferido", "Observacao"
]
for c, h in enumerate(cabs1, 1):
    hdr(ws1.cell(row=2, column=c, value=h))

segmentos = [
    ("SEG001", "Trabalhador", "26–45 anos", "M-Pesa"),
    ("SEG002", "Estudante",   "15–25 anos", "Numerário"),
    ("SEG003", "Turista",     "20–60 anos", "e-Mola"),
]
horas_pico = {
    "Trabalhador": "06h–08h / 17h–22h",
    "Estudante":   "07h–09h / 13h–15h",
    "Turista":     "09h–12h / 15h–23h"
}
observacoes = {
    "Trabalhador": "Viagens diárias regulares",
    "Estudante":   "Desconto aplicável",
    "Turista":     "Rota turística"
}

# Erros: coluna → tipo  (apenas negativo, espacos, data_errada)
erros_seg = {
    8:   (6, "negativo"),      # Total_Passageiros negativo
    21:  (2, "data_errada"),   # Data com hora
    35:  (8, "espacos"),       # Pagamento com espaços
    52:  (6, "negativo"),
    70:  (2, "data_errada"),
    91:  (8, "espacos"),
    114: (6, "negativo"),
    130: (2, "data_errada"),
}

r1 = 3
for dia in datas:
    for seg_id, tipo, faixa, pag in segmentos:
        pax = random.randint(30, 150)
        genero = random.choice(["M", "F"])
        vals = [
            seg_id, dia, tipo, genero, faixa,
            pax, horas_pico[tipo], pag, observacoes[tipo]
        ]
        if r1 in erros_seg:
            ci, te = erros_seg[r1]
            vals[ci - 1] = injectar_erro(vals[ci - 1], te)
            erros_log["Segmentos"].append(
                f"Linha {r1} | Col {cabs1[ci-1]} | {te.upper()}"
            )
        par = (r1 % 2 == 0)
        for c, v in enumerate(vals, 1):
            cell = ws1.cell(row=r1, column=c, value=v)
            dado(cell, par)
            if c == 2:
                if isinstance(v, datetime):
                    cell.number_format = "DD/MM/YYYY HH:MM"   # formato errado visível
                elif isinstance(v, date):
                    cell.number_format = "DD/MM/YYYY"
        r1 += 1

adicionar_total(ws1, r1, 5, 6, r1 - 1)
fill_total_vazio(ws1, r1, [7, 8, 9])
for col, w in zip("ABCDEFGHI", [14, 14, 16, 10, 14, 20, 22, 18, 24]):
    ws1.column_dimensions[col].width = w

wb1.save(OUTPUT_DIR + "MozChapa100_Segmentos_ERROS.xlsx")
print("✅ FILE 1 — Segmentos criado")

# ============================================================
# FILE 2 — VENDAS
# ============================================================

wb2 = openpyxl.Workbook()
ws2 = wb2.active
ws2.title = "Vendas"

adicionar_titulo(ws2, "MOZCHAPA100 — REGISTO DE VENDAS (Mar–Mai 2026)", 10)
ws2.row_dimensions[2].height = 36

cabs2 = [
    "ID_Venda", "Data", "ID_Rota", "ID_Segmento",
    "Bilhetes_Vendidos", "Preco_Unitario_MZN", "Total_MZN",
    "Forma_Pagamento", "Tipo_Pagamento", "Hora_Venda"
]
for c, h in enumerate(cabs2, 1):
    hdr(ws2.cell(row=2, column=c, value=h))

rotas_ids  = ["RT001", "RT002", "RT003", "RT004", "RT005"]
seg_ids    = ["SEG001", "SEG002", "SEG003"]
precos     = [25, 35, 50, 75, 100]
formas_pag = ["M-Pesa", "Numerário", "e-Mola"]
tipos_pag  = ["Por viagem", "Por viagem", "Por viagem", "Mensal"]
horas_venda = [
    "06:30", "07:15", "08:00", "09:30", "12:00",
    "13:45", "16:00", "17:30", "18:45", "20:00",
    "22:00", "23:30"
]

erros_venda = {
    5:   (5,  "negativo"),     # Bilhetes_Vendidos
    17:  (2,  "data_errada"),  # Data
    30:  (6,  "negativo"),     # Preco_Unitario_MZN
    48:  (8,  "espacos"),      # Forma_Pagamento
    65:  (5,  "negativo"),
    82:  (2,  "data_errada"),
    100: (6,  "negativo"),
    118: (8,  "espacos"),
    140: (5,  "negativo"),
    165: (2,  "data_errada"),
    190: (8,  "espacos"),
    215: (6,  "negativo"),
}

r2 = 3
for dia in datas:
    for _ in range(random.randint(4, 8)):
        id_v  = f"VND{r2 - 2:05d}"
        bil   = random.randint(1, 20)
        preco = random.choice(precos)
        vals  = [
            id_v, dia,
            random.choice(rotas_ids),
            random.choice(seg_ids),
            bil, preco,
            f"=E{r2}*F{r2}",
            random.choice(formas_pag),
            random.choice(tipos_pag),
            random.choice(horas_venda)
        ]
        if r2 in erros_venda:
            ci, te = erros_venda[r2]
            vals[ci - 1] = injectar_erro(vals[ci - 1], te)
            erros_log["Vendas"].append(
                f"Linha {r2} | Col {cabs2[ci-1]} | {te.upper()}"
            )
        par = (r2 % 2 == 0)
        for c, v in enumerate(vals, 1):
            cell = ws2.cell(row=r2, column=c, value=v)
            dado(cell, par)
            if c == 2:
                if isinstance(v, datetime):
                    cell.number_format = "DD/MM/YYYY HH:MM"
                elif isinstance(v, date):
                    cell.number_format = "DD/MM/YYYY"
            if c in [6, 7] and isinstance(v, (int, float)):
                cell.number_format = '#,##0.00" MZN"'
        r2 += 1

adicionar_total(ws2, r2, 6, 7, r2 - 1)
fill_total_vazio(ws2, r2, [8, 9, 10])
for col, w in zip("ABCDEFGHIJ", [14, 14, 12, 14, 18, 20, 18, 16, 14, 12]):
    ws2.column_dimensions[col].width = w

wb2.save(OUTPUT_DIR + "MozChapa100_Vendas_ERROS.xlsx")
print("✅ FILE 2 — Vendas criado")

# ============================================================
# FILE 3 — ROTAS
# ============================================================

wb3 = openpyxl.Workbook()
ws3 = wb3.active
ws3.title = "Rotas"

adicionar_titulo(ws3, "MOZCHAPA100 — REGISTO DE ROTAS (Mar–Mai 2026)", 12)
ws3.row_dimensions[2].height = 36

cabs3 = [
    "ID_Rota", "Data", "Origem", "Destino", "Distancia_KM",
    "Viagens_Realizadas", "Passageiros_Total", "Capacidade_Total",
    "Ocupacao_Pct", "Tempo_Medio_Min", "Receita_MZN", "Estado"
]
for c, h in enumerate(cabs3, 1):
    hdr(ws3.cell(row=2, column=c, value=h))

rotas_info = {
    "RT001": ("Maputo Centro", "Matola",      18, 60),
    "RT002": ("Maputo Centro", "Machava",     12, 50),
    "RT003": ("Matola",        "Boane",       25, 55),
    "RT004": ("Maputo Centro", "KaMavota",     8, 45),
    "RT005": ("Machava",       "Marracuene",  30, 60),
}
tempo_base = {"RT001": 45, "RT002": 30, "RT003": 60, "RT004": 20, "RT005": 75}

erros_rota = {
    7:   (5,  "negativo"),     # Distancia_KM
    19:  (2,  "data_errada"),  # Data
    33:  (11, "negativo"),     # Receita_MZN
    50:  (8,  "espacos"),      # Forma_Pagamento → aqui usamos Estado col 12... veja abaixo
    68:  (5,  "negativo"),
    85:  (2,  "data_errada"),
    102: (11, "negativo"),
    120: (2,  "data_errada"),
    138: (10, "negativo"),     # Tempo_Medio_Min
    157: (8,  "espacos"),      # Capacidade_Total tem string? não — usar col 6 Viagens
    175: (5,  "negativo"),
    200: (2,  "data_errada"),
}

# Nota: col 8 (Capacidade_Total) e col 6 (Viagens_Realizadas) são numéricas;
# para "espacos" em Rotas usamos col 12 (Estado) que é string.
# Ajuste correcto:
erros_rota = {
    7:   (5,  "negativo"),
    19:  (2,  "data_errada"),
    33:  (11, "negativo"),
    50:  (12, "espacos"),      # Estado — string
    68:  (5,  "negativo"),
    85:  (2,  "data_errada"),
    102: (11, "negativo"),
    120: (2,  "data_errada"),
    138: (10, "negativo"),
    157: (12, "espacos"),
    175: (5,  "negativo"),
    200: (2,  "data_errada"),
}

r3 = 3
for dia in datas:
    for rt_id, (orig, dest, dist, cap) in rotas_info.items():
        viagens  = random.randint(4, 16)
        pax      = random.randint(30, cap * viagens)
        cap_total = cap * viagens
        tempo    = tempo_base[rt_id] + random.randint(-5, 15)
        receita  = pax * random.choice([25, 35, 50])
        estado   = random.choices(["Activa", "Suspensa"], weights=[96, 4])[0]
        vals = [
            rt_id, dia, orig, dest, dist,
            viagens, pax, cap_total,
            f'=IF(H{r3}=0,"",G{r3}/H{r3})',
            tempo, receita, estado
        ]
        if r3 in erros_rota:
            ci, te = erros_rota[r3]
            vals[ci - 1] = injectar_erro(vals[ci - 1], te)
            erros_log["Rotas"].append(
                f"Linha {r3} | Col {cabs3[ci-1]} | {te.upper()}"
            )
        par = (r3 % 2 == 0)
        for c, v in enumerate(vals, 1):
            cell = ws3.cell(row=r3, column=c, value=v)
            dado(cell, par)
            if c == 2:
                if isinstance(v, datetime):
                    cell.number_format = "DD/MM/YYYY HH:MM"
                elif isinstance(v, date):
                    cell.number_format = "DD/MM/YYYY"
            if c == 9:
                cell.number_format = "0.0%"
            if c == 11 and isinstance(v, (int, float)):
                cell.number_format = '#,##0.00" MZN"'
        r3 += 1

adicionar_total(ws3, r3, 10, 11, r3 - 1)
fill_total_vazio(ws3, r3, [12])
for col, w in zip("ABCDEFGHIJKL", [12, 14, 20, 18, 14, 20, 20, 18, 14, 16, 18, 12]):
    ws3.column_dimensions[col].width = w

wb3.save(OUTPUT_DIR + "MozChapa100_Rotas_ERROS.xlsx")
print("✅ FILE 3 — Rotas criado")

# ============================================================
# SUMÁRIO
# ============================================================

total = sum(len(v) for v in erros_log.values())
print(f"\n🎉 Concluído! {total} erros injectados (3 tipos apenas)")
for f, v in erros_log.items():
    print(f"   {f}: {len(v)} erros")


✅ FILE 1 — Segmentos criado
✅ FILE 2 — Vendas criado
✅ FILE 3 — Rotas criado

🎉 Concluído! 32 erros injectados (3 tipos apenas)
   Segmentos: 8 erros
   Vendas: 12 erros
   Rotas: 12 erros


In [ ]:
# cirando uma database
conn = sqlite3.connect("mozchapa100.db")
pd.read_excel("MozChapa100_Segmentos_ERROS.xlsx", header=1).to_sql("segmentos", conn, if_exists="replace", index=False)
pd.read_excel("MozChapa100_Vendas_ERROS.xlsx",    header=1).to_sql("vendas",    conn, if_exists="replace", index=False)
pd.read_excel("MozChapa100_Rotas_ERROS.xlsx",     header=1).to_sql("rotas",     conn, if_exists="replace", index=False)
print("✅ DB criada!")

✅ DB criada!


In [ ]:
# Cctivar SQL magic
%load_ext sql
%sql sqlite:///mozchapa100.db

Connecting to 'sqlite:///mozchapa100.db'

In [ ]:
# já podes escreves SQL puro!
%%sql
SELECT *
FROM vendas
WHERE Forma_Pagamento = 'e-Mola'
LIMIT 5

Running query in 'sqlite:///mozchapa100.db'

ID_Venda,Data,ID_Rota,ID_Segmento,Bilhetes_Vendidos,Preco_Unitario_MZN,Total_MZN,Forma_Pagamento,Tipo_Pagamento,Hora_Venda
VND00003,2026-03-01 00:00:00,RT005,SEG003,-5.0,75.0,None,e-Mola,Por viagem,17:30
VND00005,2026-03-01 00:00:00,RT003,SEG001,16.0,75.0,None,e-Mola,Por viagem,18:45
VND00006,2026-03-02 00:00:00,RT004,SEG001,8.0,50.0,None,e-Mola,Por viagem,09:30
VND00013,2026-03-03 00:00:00,RT002,SEG003,6.0,75.0,None,e-Mola,Por viagem,16:00
VND00014,2026-03-03 00:00:00,RT001,SEG001,19.0,100.0,None,e-Mola,Mensal,08:00


Running query in 'sqlite:///mozchapa100.db'

++
||
++
++